# Supermarket sample (Kaggle) — Impact Split with step trace

This notebook loads [Sample Supermarket](https://www.kaggle.com/datasets/bravehart101/sample-supermarket-dataset) via `kagglehub`, fits `ImpactSplitter` with `trace=True`, and prints intermediate tables so you can confirm the algorithm behaves as expected.

## Prerequisites

- Install project deps: `pip install -r requirements.txt` (includes `kagglehub`).
- **Kaggle credentials:** for downloads, configure the Kaggle API (e.g. `~/.kaggle/kaggle.json` from your Kaggle account **Account → API → Create New Token**) or run `kagglehub.login()` once in an interactive session. See [kagglehub authentication](https://github.com/Kaggle/kagglehub#authentication).
- Target Python **3.13.x** as in `pyproject.toml`.

In [ ]:
%matplotlib inline

from pathlib import Path

import kagglehub
import pandas as pd

from impact_split import ImpactSplitter

DATASET_SLUG = "bravehart101/sample-supermarket-dataset"
root = Path(kagglehub.dataset_download(DATASET_SLUG))
csv_paths = sorted(root.rglob("*.csv"))
assert csv_paths, f"No CSV under {root}"
csv_path = csv_paths[0]
print("Using:", csv_path)

In [ ]:
df = pd.read_csv(csv_path)
df.head()

## Build `X` and `y`

- **y:** `Profit` (additive contribution at row level).
- **X:** a few categorical columns with moderate cardinality so the tree stays readable. City / postal code are omitted here.

In [ ]:
feature_cols = ["Region", "Category", "Segment", "Ship Mode"]
missing = [c for c in feature_cols + ["Profit"] if c not in df.columns]
if missing:
    raise ValueError(f"Unexpected columns: {missing}")

X = df[feature_cols].astype(str)
y = df["Profit"].astype(float)
mask = y.notna() & X.notna().all(axis=1)
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("rows:", len(X), "y.sum():", y.sum())
X.head(), y.head()

In [ ]:
model = ImpactSplitter(
    delta_pct=0.05,
    min_global_impact_pct=0.01,
    max_depth=4,
)
model.fit(X, y, trace=True)
print("trace steps:", len(model.fit_trace_))

## Trace: one row per decision node (pre-order)

Each entry includes `delta`, global materiality ratios, candidate gains per feature, and (when the node splits) category tables for every evaluated feature.

In [ ]:
trace_rows = []
for step in model.fit_trace_:
    trace_rows.append(
        {
            "node_id": step["node_id"],
            "depth": step["depth"],
            "n_samples": step["n_samples"],
            "action": step["action"],
            "stop_reason": step.get("stop_reason"),
            "delta": step["delta"],
            "materiality_pass": step["materiality_pass"],
            "chosen_feature": step.get("chosen_feature"),
            "chosen_gain": step.get("chosen_gain"),
        }
    )
pd.DataFrame(trace_rows)

In [ ]:
shown = 0
for step in model.fit_trace_:
    if step["action"] != "split":
        continue
    feat = step["chosen_feature"]
    print("=" * 80)
    print(step["node_id"], "depth", step["depth"], "split on", feat)
    cg = pd.DataFrame(step["candidate_gains"])
    print("Candidate gains:")
    display(cg)
    if feat and feat in step.get("category_tables", {}):
        print("Category assignment (chosen feature):")
        display(pd.DataFrame(step["category_tables"][feat]))
    shown += 1
    if shown >= 3:
        print("... (truncated; increase `shown` limit to inspect more splits)")
        break

In [ ]:
model.plot_tree(figsize=(14, 8))

`plot_tree` renders one matplotlib figure for the fitted tree.

In [ ]:
segments = model.get_impact_segments()
segments.head(15)

## Sanity checks

- Terminal segments partition the rows: the sum of segment `total_sum` should match the global sum of `y` (each row assigned to exactly one leaf).
- Root trace step should show `depth == 0` and global ratios `pos_ratio` / `neg_ratio` consistent with the node’s share of positive vs negative mass.
- Tightening `min_global_impact_pct` or `delta_pct` should change how soon nodes stop splitting (`stop_reason` in the trace table).

In [ ]:
seg_sum = segments["total_sum"].sum()
y_sum = float(y.sum())
print("sum(y):", y_sum)
print("sum(segment totals):", seg_sum)
assert abs(seg_sum - y_sum) < 1e-3 * max(1.0, abs(y_sum)), "Segment totals should match global y sum"